# This notebook has been adapted from 

Machine Learning for NLP - ENSAE 2022 - Lecture 2 illustration 

This notebook aims at illustrating the concepts introduced in the [lecture 2](https://nlp-ensae.github.io/files/2-ML-FOR-NLP-2022.pdf) of the Machine Learning for NLP course

Slides from https://nlp-ensae.github.io/files/2-ML-FOR-NLP-2022.pdf
(Skip to slide 17)


In [10]:
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm


# From Pixels to Words: why text isn't numbers (yet)

Last time (5.3) our mantra held up beautifully: *"pixels are numbers, everything is numbers."* An image was already an `H × W` grid of numbers, and convolution won because its shape **matched the structure of that grid**.

Text breaks the mantra. A word is a **symbol**, not a number — and our models only eat numbers (a linear layer computes `W·x + b`; you simply cannot multiply `"cat"`).

So before we reach for *any* model, we hit a problem we never had with images:

> **How do we turn a word into a vector — and can we do it so that the vector actually *means* something?**

Let's try the two most obvious fixes, watch **both of them fail**, and let those failures point us toward the real answer (word embeddings).

## Attempt 0: just feed the word in?

Our models do arithmetic on their inputs. So what happens if we try to do arithmetic on a *word*?

In [ ]:
word = "cat"

# A linear layer wants to scale its input: w * x. So... what is "half a cat"?
# TODO: try to scale the word by 0.5 (wrap in try/except to catch the error),
#       then try scaling it by 3. What does each do?
...

## Attempt 1: number the words (integer IDs)

Fine — if models want numbers, let's just make a vocabulary and hand every word an integer.

In [ ]:
# The obvious fix: build a vocabulary and give every word an integer id.
vocab_list = ["cat", "dog", "fish", "apple", "car"]
word_to_id = ...   # TODO: map each word to an integer id (hint: enumerate)
word_to_id

In [ ]:
# Everything is a number now. But look at what we *accidentally* claimed:
# TODO: check these "relationships" we never meant to create

# 1. Is 'dog' the average of 'cat' and 'fish'?
print("is 'dog' halfway between 'cat' and 'fish'? ->", ...)

# 2. Is a 'car' (4) four times a 'dog' (1)? Four dogs == one car?
print("is a 'car' == 4 x 'dog'?              ->", ...)

### That backfired 😅

Integer IDs quietly smuggled in a **fake ordering** and a **fake scale** — relationships between words that we never intended and that aren't real.

What we actually want is the opposite: numbers where **no word is "bigger than", "between", or "a multiple of" another**. Each word should stand on its own, independent of the rest.

One clean way to get exactly that: **give every word its very own dimension (its own axis).** No word is then greater than, or a blend of, any other — they're all mutually perpendicular.

That's precisely what **one-hot encoding** does — our *second* attempt. 👇

# One Hot Representations

In [11]:
vocab  = {'potato', 'petrolium', 'carrots', 'garlic', 'mac', 'basil', 'piano', 'flour'}
vecs = np.zeros((len(vocab), len(vocab)))
vecs[[range(len(vocab))], [range(len(vocab))]] = 1

potato, petrolium, carrots, garlic, mac, basil, piano, flour = vecs


In [12]:
# What is the similarity between petrolium and potato?
# What about carrots and potato?

# Similarity between vectors 🤔

## Dot Product?

$$
\textbf{a} = [a_1, a_2, \dots , a_n]\\\textbf{b} = [b_1, b_2, \dots , b_n] \\ \textbf{a}.\textbf{b} = \sum^{n}_{i=1}a_ib_i
$$

- Vectors with a larger magnitude will have higher 'score'

## Cosine?

$$
\text{cos}(\textbf{a},\textbf{b}) = \frac{\textbf{a} . \textbf{b}}{||\textbf{a}|| . ||\textbf{b}||}
$$

In [19]:
from numpy import dot
from numpy.linalg import norm

def cosine(v1, v2):
  """
  cosine similarity of two vectors
  NB: Standard metric to measure similarity between word vectors
  
  Hint: look at the imports in cell above ;)
  """
  return dot(v1, v2) / (norm(v1)*norm(v2))

a,b = np.array([1,1,1,1]), np.array([-1,2,-1,2])
print(f"Cosine of {a}, {b} = {cosine(a,b)}")

a,b = np.array([1,1,1,1]), np.array([-2,-2,-2,-2])
print(f"Cosine of {a}, {b} = {cosine(a,b)}")

a,b = np.array([1,1,1,1]), np.array([20,20,20,20])
print(f"Cosine of {a}, {b} = {cosine(a,b)}")

a,b = np.array([1,1,1,1]), np.array([-9,20,-7,-4])
print(f"Cosine of {a}, {b} = {cosine(a,b)}")


Cosine of [1 1 1 1], [-1  2 -1  2] = 0.31622776601683794
Cosine of [1 1 1 1], [-2 -2 -2 -2] = -1.0
Cosine of [1 1 1 1], [20 20 20 20] = 1.0
Cosine of [1 1 1 1], [-9 20 -7 -4] = 0.0


In [20]:
# Ok so what about our word vectors?

print(cosine(carrots, petrolium))
print(cosine(carrots, potato))

0.0
0.0


# What does it mean?

One word is completely independent from the other word. 

See and Seeing are just as apart as See and Shoe.

Is that ideal?

# More Problems with One-hot Encoding

1. Sparsity, parameters are not used properly.
2. For one word, only one column of linear transformation matrix is used.
    1. Need to learn lexical, semantic and all other notions independently for each word.

# What we want

is a way to represent a word in a way that it relates with otherwords _somehow_

# Count-Based Representations

Same dimensions as one-hot.


In [21]:

# Co-Occurences of dog, lion and car 
#  (leash,walk, run, owner, pet, barked, the)
dog =  [3,   5,   2,     5,   3,      2,  9]
lion = [0,   3,   2,     0,   1,      0,  5]
car =  [0,   0,   1,     3,   0,      0,  9]

# dog = np.array(dog)
# lion = np.array(lion)
# car =  np.array(car)

In [22]:

print(f'Example 1: Original similarities \nCosine Similarity dog vs. lion {cosine(dog, lion):.4f} ')
print(f'Cosine Similarity dog vs. car {cosine(dog, car):.4f} \n')

Example 1: Original similarities 
Cosine Similarity dog vs. lion 0.8562 
Cosine Similarity dog vs. car 0.8199 



In [26]:
# Now assume 'the dog' and 'the car' appear much more 
# (leash, walk, run, owner, pet, barked, the)
dog =  [3,   5,   2,     1,   3,      2,  9+25]
car =  [0,   0,   1,     3,   0,      0,  9+25]
# dot =  [0,   0,   2,     15,  0,      0,   1700]

print(f'Example 2: Now assume "the dog" and "the car" appear much more  \nCosine Similarity dog vs. lion {cosine(dog, lion):.4f}')
print(f'Cosine Similarity dog vs. car {cosine(dog, car):.4f} ')
print(f'What do you observe? \n')
# Frequent words impacts a lot the representations

Example 2: Now assume "the dog" and "the car" appear much more  
Cosine Similarity dog vs. lion 0.8846
Cosine Similarity dog vs. car 0.9782 
What do you observe? 



In [27]:
# Now assume 'the dog' and 'the car' appear much more 
# (leash,   walk, run, owner, pet, barked, the)
dog =  [3+1,   5,   2,     5,   3,      2,   9]
car =  [0+1,   0,   1,     3,   0,      0,   9]

print(f'Example 3: Now assume "dog leash" and "car leash" occur.  \nCosine Similarity dog vs. lion {cosine(dog, lion):.4f}')
print(f'Cosine Similarity dog vs. car {cosine(dog, car):.4f} ')
print(f'What do you observe? \n')
# Very high sensitivity to rare words

Example 3: Now assume "dog leash" and "car leash" occur.  
Cosine Similarity dog vs. lion 0.8378
Cosine Similarity dog vs. car 0.8304 
What do you observe? 



## Observations

- Count based similarity is super sensitive to less informational, very frequent word (the)
- Count based similarity is super sensitive to rare words as well (leash)

## Alternatives?

TF-IDF solves this problem. 

[Good Article explaining it](https://towardsdatascience.com/tf-idf-simplified-aba19d5f5530) | 
[Wikipedia](https://en.wikipedia.org/wiki/Tf%E2%80%93idf)

Q: **why not use tf-idf then?**

A: Its not a way to represent tokens, only a way to compute a score

# Pointwise Mutual Information

**Estimate association between events** (events = tokens, for us).
- `PMI("barack", "obama") =   20.0`: Does "Barack" often appear with "Obama" (yes → high positive value)? **they're correlated**
- `PMI("barack", "pizza") =    0.3`: Or does it appear together with "Peacock" (independent → close to zero)? **they're not correlated**
- `PMI("barack", "racist") = -13.8`: Or does it appear together with "racist"?
    - In fact no, it appears less frequently with "Barack". I.e., **they're inversely correlated**


> [PMI] is one of the most important concepts in NLP - Ch 6, Speech and Language Processing, Jurafsky and Martin


###### Let's work with this co-occurance matrix

![coccurance.png](./../resources/imgs/word-wordcooccurancematrix.png)


In [ ]:
cooccurance = np.asarray([
    [3,  5,   2,   5,     3,   2 ,    8],
    [0,  3,   2,   0,     1,   0,     6],
    [0,  0,   1,   3,     0,   0,     3]
])

#           (leash, walk, run, owner, pet, barked, the)
dog =  np.array([3,  5,   2,   5,     3,   2 ,    8])
lion = np.array([0,  3,   2,   0,     1,   0,     6])
car =  np.array([0,  0,   1,   3,     0,   0,     3])

leash =  np.array([3, 0, 0]) # 0
walk =   np.array([5, 3, 0]) # 1
run =    np.array([2, 2, 1]) # 2 ...
owner =  np.array([5, 0, 3])
pet =    np.array([3, 1, 0])
barked = np.array([2, 0, 0])
the =    np.array([8, 6, 3])


## PMI Formula
    
$$\text{pmi}(x, y) = \text{ln} \big( \frac{p(x, y)}{p(x) p(y)}\big)$$

Its a simple co-relation metric.

In [45]:
cooccurance[0, 1]

5

In [44]:
cooccurance[:, 0].sum() / cooccurance.sum()

0.06382978723404255

In [ ]:
# p(dog) = 'how many times we see dog ' / 'how many times we see anything'
#'howmmanytimes we see anything '
# cooccurance[0].sum() / cooccurance.sum()

In [46]:
def ppmi(mat, ix, iy):
    # avoid taking log of zeros; return a zero instead
    total = mat.sum()
    if total == 0:
        return 0

    p_x = mat[ix].sum() / total
    p_y = mat[:, iy].sum() / total
    p_xy = mat[ix, iy] / total

    if p_x == 0 or p_y == 0 or p_xy == 0:
        return 0
    
    return np.log(p_xy/(p_x*p_y))

In [47]:
print(ppmi(cooccurance, 1, 0)) # dog, leash
print(ppmi(cooccurance, 0, 1)) # dog, walk
print(ppmi(cooccurance, 0, 2)) # dog, run

0
0.04793946228911926
-0.3983476403393003


![PMIVEC.jpg](./../resources/imgs/pmivec.png)

In [48]:
def pmivec(cooccurance, ix):
    return np.array([ppmi(cooccurance, ix, iy) for iy in range(cooccurance.shape[1])])

In [49]:
dog_pmi = pmivec(cooccurance, 0)
lion_pmi = pmivec(cooccurance, 1)
car_pmi = pmivec(cooccurance, 2)

print(dog_pmi)
print(lion_pmi)
print(car_pmi)

[ 0.51794309  0.04793946 -0.39834764  0.04793946  0.23026102  0.51794309
 -0.23582871]
[ 0.          0.3844117   0.44895022  0.         -0.02105341  0.
  0.32378708]
[0.         0.         0.29479954 0.9234082  0.         0.
 0.1696364 ]


# Problems with all of these methods?

## Dimensionality

![NwordsStuff.jpg](./../resources/imgs/nwords_per_doc.png)

x axis - number of documents, y axis - number of unique words

With more documents, come more words. Every token will be 33k dimensions!

# Prediction-Based Representation: Word Embeddings

Let's move to notebook 6.1.1

![DocumentClassification.jpg](./../resources/imgs/onehotaintshit.jpg)

# Sources for in-depth reads

- Ch 6, Speech and Language Processing - https://web.stanford.edu/~jurafsky/slp3/6.pdf (great book; great chapter)
- Great article on understanding the intuition of PMI - https://svn.spraakdata.gu.se/repos/gerlof/pub/www/Docs/npmi-pfd.pdf